### Necessary Steps for pre-processing data

- Article Type
    - discard all except 'research-article', 'review-article', maybe 'case-report'
- Sections
    - determine which sections are necessary for inclusion (maybe determine different criteria for research vs. reviews)
    - create dict that maps each section to its standardized section name
    - discard all other papers and superfluous sections
    - clean remaining sections (remove citations?) (do this after count vectorizing when outliers become apparent)
- Date
    - reshape date format
    - sort by date 
    - re-structure the data frames, such that each data frame contains all papers from a given year
- Language
    - infer langugage for all papers without specified language
    - discard all non-english



In [1]:
import pandas as pd
import re
import os
from collections import Counter
from polyglot.detect import Detector
from polyglot.detect.base import Error

%load_ext autoreload
%autoreload 2
import src.data_utils as utils

In [2]:
BASELINE_NAME = 'baseline_2025-06-26'

In [3]:
baseline_df = pd.read_json('../data/' + BASELINE_NAME + '/PMC008xxxxxx_noncomm.json')

In [4]:
baseline_df.head()

,article-type,language,journal,pmc-id,pmid,title,country,date,abstract,section_titles,sections
0,correction,None,Diabetes Therapy,PMC8991299,35286609.0,Correction to: Overt Diabetes in Pregnancy,India,14-3-2022,,[Correction to: Diabetes Ther 10.1007/s13300-0...,[Correction to: Diabetes Ther 10.1007/s13300-0...
1,research-article,en,International Journal of Chronic Obstructive P...,PMC8749770,35027824.0,Effect of Smoking and Its Cessation on the Tra...,India,05-1-2022,Rationale Smoking is the primary cause of chro...,"[Introduction, Methods, Results, Discussion, C...",[Introduction Chronic Obstructive Pulmonary Di...
2,research-article,en,Journal of Clinical Laboratory Analysis,PMC8605134,34528313.0,Evaluation of fungal contamination and ochrato...,"Iran, Islamic Republic of",15-9-2021,Abstract Background Mycotoxins are secondary f...,"[INTRODUCTION, MATERIALS, RESULT, DISCUSSION, ...",[1 INTRODUCTION Coffee is a member of Coffea g...
3,review-article,en,International Journal of Clinical Pediatric De...,PMC8783216,NaN,The Epiphany of Post-COVID: A Watershed for Pe...,India,Nov-Dec-2021,A bstract Coronavirus disease-2019 (COVID-19) ...,"[I, E, T, S, D, C]",[I ntroduction Coronavirus disease-2019 (COVID...
4,research-article,None,Schizophrenia Research: Cognition,PMC8131976,34026572.0,Screening for cognitive impairment in schizoph...,Austria,12-5-2021,Background The Screen for Cognitive Impairment...,"[Introduction, Methods and materials, Results,...",[1 Introduction Cognitive impairment is consid...


In [5]:
paper_count_0 = len(baseline_df)
paper_count_0

185530

#### Article Type

In [6]:
include_types = ['research-article', 'review-article', 'case-report']
baseline_df = baseline_df[baseline_df['article-type'].apply(lambda x: x in include_types)]
len(baseline_df)

154669

#### Sections

In [7]:
#replace None with empty string to make my life easier
baseline_df['section_titles'] = baseline_df['section_titles'].apply(
    lambda x: [] if x == None else x
)

In [8]:
def standardize_sections(sections: list, section_titles: list) -> list:
    """replaces list entries of 'sections' with dict entries, where each section is mapped to its standardized section name.
    sections without a standardized name are discarded
    if there is more than one section with the same standardized name (e.g. 'case 1' and 'case 2' would both be assigned to 
    'case report'), the section strings are concatenated 
    Arguments:
    - sections: nested list, each entry a list of strings, each is a section from a paper
    - section_titles: same as sections, but with corresponding title for each section
    """
    
    
    standardized = []
    
    for i, secs in enumerate(sections):
        sec_dict = {}
        if len(secs) == len(section_titles[i]):
            for j, sec in enumerate(secs):
                alt_title = utils.get_alt_title(section_titles[i][j])
                if not alt_title == '':
                    if alt_title in sec_dict.keys():
                        sec_dict[alt_title] += ' ' + sec
                    else:
                        sec_dict[alt_title] = sec
        standardized.append(sec_dict)
        

    return standardized

In [9]:
test_secs = [['this is an intro.', 'these are methods.', 'this is a result.'],
             ['bla bla bla', 'this is the first method', 'bla bla bla bla', 'this is the second method'],
             ['this is being ignored']]

test_titles = [['introduction', 'method', 'result'],
               ['bla', 'method', 'bla', 'method'],
               []]

standardize_sections(test_secs, test_titles)

[{'introduction': 'this is an intro.',
  'methods': 'these are methods.',
  'results': 'this is a result.'},
 {'methods': 'this is the first method this is the second method'},
 {}]

In [10]:
baseline_df['sections'] = standardize_sections(list(baseline_df['sections']), list(baseline_df['section_titles']))
baseline_df.head()

,article-type,language,journal,pmc-id,pmid,title,country,date,abstract,section_titles,sections
1,research-article,en,International Journal of Chronic Obstructive P...,PMC8749770,35027824.0,Effect of Smoking and Its Cessation on the Tra...,India,05-1-2022,Rationale Smoking is the primary cause of chro...,"[Introduction, Methods, Results, Discussion, C...",{'introduction': 'Introduction Chronic Obstruc...
2,research-article,en,Journal of Clinical Laboratory Analysis,PMC8605134,34528313.0,Evaluation of fungal contamination and ochrato...,"Iran, Islamic Republic of",15-9-2021,Abstract Background Mycotoxins are secondary f...,"[INTRODUCTION, MATERIALS, RESULT, DISCUSSION, ...",{'introduction': '1 INTRODUCTION Coffee is a m...
3,review-article,en,International Journal of Clinical Pediatric De...,PMC8783216,NaN,The Epiphany of Post-COVID: A Watershed for Pe...,India,Nov-Dec-2021,A bstract Coronavirus disease-2019 (COVID-19) ...,"[I, E, T, S, D, C]",{}
4,research-article,None,Schizophrenia Research: Cognition,PMC8131976,34026572.0,Screening for cognitive impairment in schizoph...,Austria,12-5-2021,Background The Screen for Cognitive Impairment...,"[Introduction, Methods and materials, Results,...",{'introduction': '1 Introduction Cognitive imp...
6,research-article,en,Cancer Research,PMC8448979,34341073.0,PAX9 Determines Epigenetic State Transition an...,None,02-8-2021,A genome-wide screen in small cell lung cancer...,"[Introduction, Materials and Methods, Results,...",{'introduction': 'Introduction Lung cancer is ...


In [11]:
# remove papers with empty sections dictionaries
baseline_df = baseline_df[baseline_df['sections'].apply(lambda x: not x == {})]
len(baseline_df)

145361

#### Language

In [12]:
Detector(baseline_df['abstract'].iloc[0]).languages[0].code

'en'

In [13]:
Detector('abstracts are an important part of papers').languages[0].code

'en'

In [14]:
def determine_lang(langs: list, texts: list):
    
    langs_new = []

    for i, lang in enumerate(langs):
        if lang == None and not texts[i] == None:
            try:
                lang = Detector(texts[i]).languages[0].code
            except Error:
                lang = None
        
        langs_new.append(lang)
    
    return langs_new

In [15]:
baseline_df['language'] = determine_lang(baseline_df['language'], list(baseline_df['abstract']))

Detector is not able to detect the language reliably.
Detector is not able to detect the language reliably.
Detector is not able to detect the language reliably.
Detector is not able to detect the language reliably.


In [16]:
# remove non-english entries
baseline_df = baseline_df[baseline_df['language'].apply(lambda x: x == 'en')]
len(baseline_df)

139279

#### Dates

In [12]:
# for the seasons, the meterological start of the season is used as the date 
# for time spans, the first date is used (although the span can be huge, e.g. 'Jan-Jun-2021' for PMC8297570)
alt_dates = {'Jan': '01-01',
             'Feb': '01-02',
             'Mar': '01-03',
             'Apr': '01-04',
             'May': '01-05',
             'Jun': '01-06',
             'Jul': '01-07',
             'Aug': '01-08',
             'Sep': '01-09',
             'Oct': '01-10',
             'Nov': '01-11',
             'Dec': '01-12',
             'Spring': '01-03',
             'Summer': '01-06',
             'Fall': '01-09',
             'Winter': '01-12'}

In [13]:
dates_formatted = []
for i, date in enumerate(baseline_df['date']):
    # if the date starts with letters, save them to get the alt_date entry
    r = r'^[a-zA-Z]*'
    key = re.match(r, date).group()
    # replace letters with correct digits
    if not key == '':
        r = r'.+(?=\-\d{4})'
        date = re.sub(r, alt_dates[key], date)
    # remove everything after year
    r = r'(?<=\-\d{4}).+'
    date = re.sub(r, '', date)

    try: 
        date = pd.to_datetime(date, dayfirst = True)
    except pd._libs.tslibs.parsing.DateParseError:
        print(f'{date} with pmc-id {baseline_df['pmc-id'].iloc[i]}')
    
    dates_formatted.append(date)

In [14]:
baseline_df['date'] = dates_formatted


In [15]:
baseline_df = baseline_df.sort_values('date', ignore_index=True)

re-organize json files (in a new folder), such that there is a json file for each year

In [26]:
os.makedirs('../data/' + BASELINE_NAME + '/formatted', exist_ok=True)

In [27]:
for year in range(2000, 2026):
    df = baseline_df[baseline_df['date'].apply(lambda x: x.year == year)]

    # check if there is already a df for the current year
    json_path = os.path.join('../data', BASELINE_NAME, 'formatted', str(year) + '.json')
    if os.path.exists(json_path):
        df_old = pd.read_json(json_path)
        df = pd.concat([df_old, df], ignore_index=True).drop_duplicates(subset=['pmc-id'])
    
    # save df as json
    if len(df) > 0:
        df.to_json(json_path)
    



In [28]:
df = pd.read_json(os.path.join('../data', BASELINE_NAME, 'formatted', '2000.json'))
df

,article-type,language,journal,pmc-id,pmid,title,country,date,abstract,section_titles,sections
0,research-article,NaN,Nature,PMC8288016,10952301,DNA sequence of both chromosomes of the choler...,United States of America,2000-08-01,Here we determine the complete genomic sequenc...,"[Main, Genome analysis, Comparative genomics, ...",{'conclusion': 'Conclusions The Vibrio choler...
